# 🍷 Ejercicio 3 – RAG: El sumiller de La Trattoria

En los ejercicios anteriores el camarero Marco podía consultar la carta de platos. Pero si un cliente le pregunta qué vino marida mejor con el carpaccio... ¡no tiene ni idea!

En este ejercicio vamos a construir un sistema **RAG** que indexa la carta de vinos del restaurante y permite hacer búsquedas semánticas sobre ella.

---

## ¿Qué es RAG?

**RAG** (Retrieval-Augmented Generation) es una técnica que combina dos pasos:

```
         INDEXACIÓN (se hace una vez)
         ┌─────────────────────────────────────┐
         │  Documento  →  Fragmentos  →  FAISS │
         └─────────────────────────────────────┘

         CONSULTA (se hace cada vez que el usuario pregunta)
         ┌──────────────────────────────────────────────────────────┐
         │  Pregunta  →  Buscar en FAISS  →  Fragmentos relevantes  │
         │            →  LLM (pregunta + fragmentos)  →  Respuesta  │
         └──────────────────────────────────────────────────────────┘
```

La clave es que el LLM **no memoriza** el documento: lo consulta en el momento en que lo necesita.

---

## ¿Qué es un embedding?

Un **embedding** es una representación numérica (un vector) de un texto. Textos con significados parecidos tienen vectores parecidos, aunque usen palabras distintas.

Por ejemplo:
- *"vino tinto con cuerpo"* y *"red wine full-bodied"* → vectores muy cercanos
- *"vino tinto"* y *"helado de vainilla"* → vectores muy lejanos

**Sentence Transformers** es la librería que convierte texto en embeddings.  
**FAISS** es la librería que almacena esos vectores y permite buscar los más cercanos a una consulta de forma muy eficiente.

> 💡 No necesitas GPU ni un ordenador potente para este ejercicio. El modelo que usamos pesa solo ~120MB y FAISS funciona perfectamente en CPU para colecciones pequeñas.

## Paso 1 – Instalación de dependencias

In [4]:
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv

## Paso 2 – Configuración de la API Key

In [5]:
import os
from dotenv import load_dotenv

#load_dotenv()
from google.colab import userdata
api_key = userdata.get('GROQ_API_KEY')
# Si no tienes fichero .env, descomenta y pon tu clave:
# os.environ["GROQ_API_KEY"] = "TU_API_KEY_AQUI"

print("✅ API Key configurada")

✅ API Key configurada


## Paso 3 – Dividir el documento en fragmentos

El fichero `vinos.txt` ya está en tu directorio. Vamos a leerlo y dividirlo en **chunks**: trozos de texto que se indexarán de forma independiente.

Una estrategia sencilla es dividir por líneas en blanco — en nuestro fichero, cada vino es un bloque separado.

> 💡 El tamaño del chunk es importante: chunks muy cortos pierden contexto, chunks muy largos incluyen información irrelevante en la búsqueda.

In [6]:
with open("/content/drive/MyDrive/vinos.txt", "r", encoding="utf-8") as f:
    contenido = f.read()

# Dividimos por bloques separados por línea en blanco y filtramos vacíos
chunks = [c.strip() for c in contenido.split("\n\n") if c.strip()]

print(f"✅ Documento dividido en {len(chunks)} fragmentos")
print()
print("Primer fragmento:")
print(chunks[0])

✅ Documento dividido en 17 fragmentos

Primer fragmento:
LA TRATTORIA – CARTA DE VINOS


## Paso 4 – Generar embeddings con Sentence Transformers

Convertimos cada fragmento de texto en un vector numérico usando un modelo de embeddings.

Usamos `paraphrase-multilingual-MiniLM-L12-v2`: un modelo multilingüe (~120MB) entrenado en 50+ idiomas incluyendo español. Rápido en CPU y con excelente calidad para búsquedas semánticas en castellano.

In [7]:
from sentence_transformers import SentenceTransformer

# Cargamos el modelo (se descarga automáticamente la primera vez, ~120MB)
modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Generamos un embedding por cada chunk
embeddings = modelo_embeddings.encode(chunks, show_progress_bar=True)

print(f"\n✅ Embeddings generados")
print(f"   Número de vectores: {embeddings.shape[0]}")
print(f"   Dimensiones por vector: {embeddings.shape[1]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embeddings generados
   Número de vectores: 17
   Dimensiones por vector: 384


## Paso 5 – Crear el índice FAISS

FAISS almacena los vectores y permite buscar los más similares a una consulta de forma muy eficiente.

Usamos `IndexFlatL2`: el índice más simple, que calcula la distancia euclídea entre vectores. Es perfecto para colecciones pequeñas.

In [8]:
import faiss
import numpy as np

# FAISS necesita los vectores en formato float32
embeddings_np = np.array(embeddings).astype("float32")

# Creamos el índice con la dimensión de nuestros embeddings
dimension = embeddings_np.shape[1]
indice = faiss.IndexFlatL2(dimension)

# Añadimos los embeddings al índice
indice.add(embeddings_np)

print(f"✅ Índice FAISS creado")
print(f"   Vectores indexados: {indice.ntotal}")

✅ Índice FAISS creado
   Vectores indexados: 17


## Paso 6 – Búsqueda semántica

Ahora podemos buscar los fragmentos más relevantes para una pregunta.

El proceso es:
1. Convertir la pregunta en un embedding (con el mismo modelo)
2. Buscar en FAISS los vectores más cercanos
3. Recuperar los chunks de texto correspondientes

Fíjate en que la búsqueda es **semántica**: no busca palabras exactas, sino significados similares.

In [9]:
def buscar(pregunta: str, top_k: int = 2) -> list[str]:
    """Busca los fragmentos más relevantes para una pregunta."""
    # Convertimos la pregunta en un embedding
    embedding_pregunta = modelo_embeddings.encode([pregunta]).astype("float32")

    # Buscamos los top_k fragmentos más cercanos en FAISS
    distancias, indices = indice.search(embedding_pregunta, top_k)

    # Recuperamos los textos correspondientes
    resultados = [chunks[i] for i in indices[0]]
    return resultados

# Probamos la búsqueda
pregunta = "¿Qué vino va bien con el pescado?"
resultados = buscar(pregunta)

print(f"🔍 Pregunta: {pregunta}")
print()
for i, fragmento in enumerate(resultados, 1):
    print(f"--- Fragmento {i} ---")
    print(fragmento)
    print()

🔍 Pregunta: ¿Qué vino va bien con el pescado?

--- Fragmento 1 ---
Vermentino di Sardegna – Sella & Mosca, Cerdeña (Italia)
Uva: Vermentino. Fermentación en acero inoxidable.
Aromas de melocotón, hinojo, almendra y una característica nota salina. Seco, con acidez equilibrada y un ligero amargor final típico de la variedad.
Maridaje: excepcional con pescados a la plancha, carpaccio de lubina y la bruschetta al pomodoro.
Copa: 7€ | Botella: 32€

--- Fragmento 2 ---
Prosecco di Valdobbiadene DOCG – Nino Franco, Véneto (Italia)
Uva: Glera. Método Charmat.
Burbujas finas y persistentes. Notas de manzana golden, pera y flores blancas. Seco (brut), ligero y refrescante.
Maridaje: el espumoso de bienvenida por excelencia. Ideal como aperitivo, con frituras ligeras y la bruschetta. También armoniza con el tiramisú.
Copa: 6€ | Botella: 26€



## Paso 7 – Conectar con el LLM: el sumiller completo

Hasta ahora solo recuperamos fragmentos. El último paso es pasar esos fragmentos al LLM junto con la pregunta para que genere una respuesta en lenguaje natural.

Este es el pipeline RAG completo:

```
Pregunta → FAISS → Fragmentos relevantes → LLM → Respuesta
```

In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, api_key=api_key)

def preguntar_al_sumiller(pregunta: str) -> str:
    """Pipeline RAG completo: busca fragmentos relevantes y genera una respuesta."""

    # 1. Recuperar fragmentos relevantes
    fragmentos = buscar(pregunta, top_k=2)
    contexto = "\n\n".join(fragmentos)

    # 2. Construir el prompt con el contexto recuperado
    mensajes = [
        SystemMessage(content=(
            "Eres Luca, el sumiller del restaurante italiano 'La Trattoria'. "
            "Respondes en español con un tono elegante y apasionado por el vino. "
            "Basa tus respuestas ÚNICAMENTE en la información de la carta de vinos que se te proporciona. "
            "Si la información no está en la carta, dilo con honestidad."
        )),
        HumanMessage(content=(
            f"Carta de vinos disponible:\n{contexto}\n\n"
            f"Pregunta del cliente: {pregunta}"
        )),
    ]

    # 3. Generar la respuesta
    respuesta = llm.invoke(mensajes)
    return respuesta.content

# Probamos el sumiller completo
pregunta = "¿Qué vino me recomiendas para acompañar un carpaccio de ternera?"
print(f"👤 Cliente: {pregunta}")
print(f"🍷 Luca: {preguntar_al_sumiller(pregunta)}")

👤 Cliente: ¿Qué vino me recomiendas para acompañar un carpaccio de ternera?
🍷 Luca: Excelente elección el carpaccio de ternera. Para acompañarlo, te recomiendo sin duda el Franciacorta Brut de Bellavista. Este vino espumoso, elaborado con Chardonnay y Pinot Nero, ofrece una burbuja finísima y aromas de bollería, cítricos y brioche que complementarán perfectamente la delicadeza del carpaccio.

Su gran complejidad y elegancia lo hacen ideal para acompañar platos como el carpaccio, y según nuestra carta de vinos, es una excelente opción para maridar con este plato en particular. Además, su crianza de 24 meses sobre lías le aporta una riqueza y profundidad que realzarán la experiencia gastronómica.

Puedes disfrutar de una copa de Franciacorta Brut por 10€ o, si lo prefieres, puedes pedir una botella por 52€. ¿Te gustaría probarlo?


---

## 🏋️ Retos para practicar

### Reto 1 – Inspecciona las distancias FAISS

La función `buscar()` descarta las distancias que devuelve `indice.search()`. Crea una versión extendida que las muestre junto a cada fragmento: cuanto menor sea la distancia, más relevante es el resultado.

Prueba con preguntas muy distintas. ¿Cómo cambian los valores de distancia?

In [11]:
def buscar_con_distancias(pregunta: str, top_k: int = 2):
    embedding_pregunta = modelo_embeddings.encode([pregunta]).astype("float32")
    distancias, indices = indice.search(embedding_pregunta, top_k)
    print(f"🔍 Pregunta: {pregunta}\n")
    for rank, (idx, dist) in enumerate(zip(indices[0], distancias[0]), 1):
        # Imprime la distancia y el fragmento correspondiente
        print(f"--- Fragmento {rank} (Distancia: {dist:.4f}) ---")
        print(chunks[idx])
        print()

buscar_con_distancias("¿Qué vino va bien con la pasta?")
# Prueba con otra pregunta diferente
buscar_con_distancias("Quiero un vino dulce para los postres")

🔍 Pregunta: ¿Qué vino va bien con la pasta?

--- Fragmento 1 (Distancia: 17.1289) ---
Franciacorta Brut – Bellavista, Lombardía (Italia)
Uva: Chardonnay 70%, Pinot Nero 30%. Método tradicional (segunda fermentación en botella). Crianza: 24 meses sobre lías.
El equivalente italiano del Champagne. Burbuja finísima, aromas de bollería, cítricos y brioche. Gran complejidad y elegancia.
Maridaje: a la altura de cualquier ocasión especial. Marida con toda la carta, especialmente con el carpaccio, la panna cotta y el gelato artesanal.
Copa: 10€ | Botella: 52€

--- Fragmento 2 (Distancia: 17.7633) ---
Prosecco di Valdobbiadene DOCG – Nino Franco, Véneto (Italia)
Uva: Glera. Método Charmat.
Burbujas finas y persistentes. Notas de manzana golden, pera y flores blancas. Seco (brut), ligero y refrescante.
Maridaje: el espumoso de bienvenida por excelencia. Ideal como aperitivo, con frituras ligeras y la bruschetta. También armoniza con el tiramisú.
Copa: 6€ | Botella: 26€

🔍 Pregunta: Quiero un vi

### Reto 2 – Cambia el número de fragmentos recuperados

Prueba la misma pregunta con `top_k=1`, `top_k=2` y `top_k=4` y compara las respuestas del sumiller.

¿Mejoran con más contexto? ¿A partir de qué valor el ruido empieza a perjudicar la respuesta?

In [12]:
def preguntar_al_sumiller_variable(pregunta: str, top_k: int) -> str:
    fragmentos = buscar(pregunta, top_k=top_k)
    contexto = "\n\n".join(fragmentos)

    mensajes = [
        SystemMessage(content=(
            "Eres Luca, el sumiller del restaurante italiano 'La Trattoria'. "
            "Respondes en español con un tono elegante y apasionado. "
            "Basa tus respuestas ÚNICAMENTE en la información proporcionada. "
            "Si la información no está, dilo con honestidad."
        )),
        HumanMessage(content=(f"Carta de vinos:\n{contexto}\n\nPregunta: {pregunta}")),
    ]
    return llm.invoke(mensajes).content

pregunta = "¿Qué vino me recomiendas para una cena romántica?"

for k in [1, 2, 4]:
    print(f"\n🍷 --- Probando con top_k = {k} ---")
    print(preguntar_al_sumiller_variable(pregunta, top_k=k))


🍷 --- Probando con top_k = 1 ---
Una cena romántica es el escenario perfecto para un vino que sea elegante, refinado y capaz de complementar la magia del momento. En nuestra carta de vinos, tenemos un espumoso que creo que sería ideal para esta ocasión: el Prosecco di Valdobbiadene DOCG de Nino Franco.

Este vino, con su método Charmat y uva Glera, ofrece una experiencia sensorial única. Sus burbujas finas y persistentes, junto con notas de manzana golden, pera y flores blancas, crearán un ambiente de romance y sofisticación en su cena. Además, su carácter seco y refrescante lo hace perfecto para acompañar una variedad de platos, desde aperitivos ligeros hasta postres como el tiramisú.

Imagínese brindando con una copa de este delicioso Prosecco mientras disfruta de una velada íntima con su pareja. El precio de 6€ por copa o 26€ por botella lo hace accesible y una excelente opción para hacer de su cena romántica un recuerdo inolvidable. ¿Qué le parece si empezamos con una copa de este

### Reto 3 – ¿Qué pasa con preguntas fuera de la carta?

Haz preguntas sobre cosas que no existen en `vinos.txt` (por ejemplo, pedir un whisky o una cerveza). FAISS siempre devolverá algo — la cuestión es qué y con qué distancia.

¿Las distancias son mayores o menores que en preguntas relevantes? ¿El sumiller lo reconoce o inventa?

In [13]:
preguntas_fuera = [
    "¿Tenéis alguna cerveza artesanal?",
    "Me gustaría pedir un whisky escocés con hielo, ¿qué tenéis?"
]

for pregunta in preguntas_fuera:
    print("-" * 50)
    # 1. Vemos las distancias de FAISS
    buscar_con_distancias(pregunta)

    # 2. Vemos cómo lo gestiona el LLM
    print(f"🍷 Luca: {preguntar_al_sumiller(pregunta)}\n")

--------------------------------------------------
🔍 Pregunta: ¿Tenéis alguna cerveza artesanal?

--- Fragmento 1 (Distancia: 21.7788) ---
Franciacorta Brut – Bellavista, Lombardía (Italia)
Uva: Chardonnay 70%, Pinot Nero 30%. Método tradicional (segunda fermentación en botella). Crianza: 24 meses sobre lías.
El equivalente italiano del Champagne. Burbuja finísima, aromas de bollería, cítricos y brioche. Gran complejidad y elegancia.
Maridaje: a la altura de cualquier ocasión especial. Marida con toda la carta, especialmente con el carpaccio, la panna cotta y el gelato artesanal.
Copa: 10€ | Botella: 52€

--- Fragmento 2 (Distancia: 23.3865) ---
Primitivo di Manduria – Feudi di San Marzano, Puglia (Italia)
Uva: Primitivo. Sin crianza en madera.
Vino de cuerpo generoso, con aromas de mora, higo y especias dulces. Suave en boca, con final cálido y persistente.
Maridaje: marida bien con embutidos, bruschetta y quesos semicurados. También con pasta arrabbiata por su carácter frutal.
Copa: 

### Reto 4 – Búsqueda en otro idioma

El modelo está entrenado en 50+ idiomas. Prueba a hacer preguntas en inglés con la carta en español. ¿Recupera los fragmentos correctos? ¿Las distancias son similares a las de la misma pregunta en español?

In [14]:
# Comparamos distancias y resultados en inglés y español
print("--- VERSIÓN EN INGLÉS ---")
buscar_con_distancias("What wine goes well with fish?")

print("--- VERSIÓN EN ESPAÑOL ---")
buscar_con_distancias("¿Qué vino va bien con el pescado?")

--- VERSIÓN EN INGLÉS ---
🔍 Pregunta: What wine goes well with fish?

--- Fragmento 1 (Distancia: 26.3939) ---
Vermentino di Sardegna – Sella & Mosca, Cerdeña (Italia)
Uva: Vermentino. Fermentación en acero inoxidable.
Aromas de melocotón, hinojo, almendra y una característica nota salina. Seco, con acidez equilibrada y un ligero amargor final típico de la variedad.
Maridaje: excepcional con pescados a la plancha, carpaccio de lubina y la bruschetta al pomodoro.
Copa: 7€ | Botella: 32€

--- Fragmento 2 (Distancia: 27.2722) ---
Prosecco di Valdobbiadene DOCG – Nino Franco, Véneto (Italia)
Uva: Glera. Método Charmat.
Burbujas finas y persistentes. Notas de manzana golden, pera y flores blancas. Seco (brut), ligero y refrescante.
Maridaje: el espumoso de bienvenida por excelencia. Ideal como aperitivo, con frituras ligeras y la bruschetta. También armoniza con el tiramisú.
Copa: 6€ | Botella: 26€

--- VERSIÓN EN ESPAÑOL ---
🔍 Pregunta: ¿Qué vino va bien con el pescado?

--- Fragmento 1 (D

---

## 🏆 Reto final – El sumiller como tool del agente

Hasta ahora el sumiller y el camarero son sistemas separados. En este reto los vas a unir: convierte el pipeline RAG en una **tool** y dásela al agente del Ejercicio 2, para que Marco pueda responder tanto preguntas sobre la carta de platos como recomendaciones de vino.

Pistas:
- Crea una tool `consultar_sumiller(pregunta: str) -> str` que internamente ejecute el pipeline RAG completo (buscar en FAISS + llamar al LLM) y devuelva la respuesta de Luca.
- Pásale esta tool al agente junto con `leer_carta`, de forma que tenga acceso a ambas.
- Ajusta el `system_prompt` del agente para que sepa que ahora tiene acceso tanto a la carta de platos como a la carta de vinos, y que puede recomendar maridajes.
- Prueba con preguntas que mezclen platos y vinos: *"Voy a pedir la tagliata di manzo, ¿qué vino me recomiendas?"*

In [15]:
%pip install -U langchain langchain-community

In [20]:
# ✏️ Tu código aquí
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# 1. Creamos la Tool del Sumiller (RAG)
@tool
def consultar_sumiller(pregunta: str) -> str:
    """
    Útil para consultar la carta de vinos del restaurante y pedir recomendaciones de maridaje.
    """
    # Usamos la función buscar() definida en pasos anteriores del cuaderno
    fragmentos = buscar(pregunta, top_k=2)
    contexto = "\n\n".join(fragmentos)

    mensajes = [
        SystemMessage(content="Eres Luca, sumiller. Basa tu respuesta en la carta de vinos:\n" + contexto),
        HumanMessage(content=pregunta)
    ]
    return llm.invoke(mensajes).content

# 2. Tool simulada de la carta de comida (del Ejercicio 2)
@tool
def leer_carta(pregunta: str) -> str:
    """Útil para consultar los ingredientes o platos de la carta de comida."""
    return "La tagliata di manzo es un exquisito corte de ternera servido con rúcula y lascas de parmesano."

# 3. Unimos las herramientas al LLM directamente
tools = [leer_carta, consultar_sumiller]
llm_con_tools = llm.bind_tools(tools)

# 4. Hacemos la prueba con el bucle de razonamiento
pregunta_prueba = "Voy a pedir la tagliata di manzo, ¿qué vino me recomiendas?"
print(f"👤 Cliente: {pregunta_prueba}\n")

mensajes_agente = [
    SystemMessage(content=(
        "Eres Marco, camarero de La Trattoria. Usa las herramientas necesarias. "
        "Si te preguntan por un plato y no sabes qué es, usa primero 'leer_carta'. "
        "Luego usa 'consultar_sumiller' para buscar el maridaje. Finalmente, responde al cliente de forma amable."
    )),
    HumanMessage(content=pregunta_prueba)
]

# Bucle del agente: seguirá iterando mientras necesite usar herramientas
while True:
    respuesta_llm = llm_con_tools.invoke(mensajes_agente)
    mensajes_agente.append(respuesta_llm)

    # Si el LLM no devuelve "tool_calls", significa que ha terminado y nos da la respuesta final en texto
    if not hasattr(respuesta_llm, 'tool_calls') or not respuesta_llm.tool_calls:
        print(f"\n🍝 Marco: {respuesta_llm.content}")
        break # Salimos del bucle

    # Si quiere usar herramientas, las ejecutamos una a una
    for tool_call in respuesta_llm.tool_calls:
        print(f"⚙️  [Marco está pensando... usando la herramienta: {tool_call['name']}]")

        if tool_call["name"] == "leer_carta":
            resultado = leer_carta.invoke(tool_call["args"])
        elif tool_call["name"] == "consultar_sumiller":
            resultado = consultar_sumiller.invoke(tool_call["args"])
        else:
            resultado = "No se ha encontrado la herramienta."

        # Añadimos la respuesta de la herramienta al historial para que Marco la lea en el siguiente ciclo
        mensajes_agente.append(ToolMessage(content=resultado, tool_call_id=tool_call["id"]))

👤 Cliente: Voy a pedir la tagliata di manzo, ¿qué vino me recomiendas?

⚙️  [Marco está pensando... usando la herramienta: leer_carta]
⚙️  [Marco está pensando... usando la herramienta: consultar_sumiller]
⚙️  [Marco está pensando... usando la herramienta: consultar_sumiller]

🍝 Marco: Lo siento, pero no tengo suficiente información para recomendarte un vino específico para acompañar la tagliata di manzo. Te sugiero consultar con un sumiller o un experto en vinos para obtener una recomendación más precisa.


---

## 📝 Reflexión final

Responde en la celda de abajo (como comentarios):

1. ¿En qué se diferencia RAG de simplemente meter todo el documento en el prompt del LLM?
2. ¿Qué ventaja tiene la búsqueda semántica frente a una búsqueda por palabras clave (como `ctrl+F`)?
3. ¿Qué pasaría si el documento fuera muy grande (miles de páginas)? ¿Seguiría funcionando este enfoque?
4. ¿Para qué otros casos de uso de una empresa real crees que sería útil RAG?

In [21]:
# ✏️ Escribe aquí tus reflexiones

# 1. RAG inyecta solo los fragmentos más relevantes, ahorrando tokens, reduciendo costes y evitando que el modelo se confunda u "olvide" datos por exceso de texto.

# 2. Entiende el contexto y los sinónimos (relaciona "pescado" con "lubina" o "mar", por ejemplo), mientras que la búsqueda tradicional exige coincidencias exactas.

# 3. Sí, funcionaría perfectamente. FAISS filtra millones de vectores en milisegundos, por lo que el LLM siempre recibirá un prompt corto y procesará la respuesta rápido.

# 4. Es ideal para chatbots de soporte técnico (buscar en manuales), asistentes legales (jurisprudencia) o herramientas de recursos humanos (políticas internas).